In [ ]:
# ============================================================
# 03_EXPERIENCE_EXTRACTION.IPYNB
# ============================================================

import pandas as pd
import numpy as np
import re

# ============================================================
# STEP 1 : LOAD SKILL EXTRACTED DATASET
# ============================================================

df = pd.read_csv("skill_extracted_resumeJD.csv")

print("="*50)
print("DATASET LOADED")
print("="*50)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

# ============================================================
# STEP 2 : RESUME EXPERIENCE EXTRACTION
# ============================================================

def extract_resume_experience(text):

    text = str(text).lower()

    patterns = [

        r'(\d+)\+?\s*years',
        r'(\d+)\+?\s*year',
        r'(\d+)\s*yrs',
        r'(\d+)\s*yr'

    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            return int(match.group(1))

    return np.nan

# ============================================================
# STEP 3 : JD EXPERIENCE EXTRACTION
# ============================================================

def extract_jd_experience(text):

    text = str(text).lower()

    patterns = [

        r'experience\s*:?\s*(\d+)\+?\s*years',
        r'experience\s*:?\s*(\d+)\+?\s*year',
        r'(\d+)\+?\s*years',
        r'(\d+)\+?\s*year'

    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            return int(match.group(1))

    return np.nan

# ============================================================
# STEP 4 : CREATE EXPERIENCE COLUMNS
# ============================================================

df["resume_experience"] = df["resume_text"].apply(
    extract_resume_experience
)

df["jd_experience"] = df["job_description"].apply(
    extract_jd_experience
)

print("\nExperience Extraction Completed")

# ============================================================
# STEP 5 : EXPERIENCE MATCH SCORE
# ============================================================

def calculate_experience_match_score(
        resume_exp,
        jd_exp):

    if pd.isna(resume_exp):
        return np.nan

    if pd.isna(jd_exp):
        return np.nan

    if jd_exp == 0:
        return 0

    score = resume_exp / jd_exp

    return round(
        min(score, 1),
        2
    )

# ============================================================
# STEP 6 : GENERATE EXPERIENCE SCORE
# ============================================================

df["experience_match_score"] = df.apply(

    lambda row:
    calculate_experience_match_score(

        row["resume_experience"],
        row["jd_experience"]

    ),

    axis=1

)

print("Experience Match Score Generated")

# ============================================================
# STEP 7 : SAMPLE OUTPUT
# ============================================================

print("\n" + "="*50)
print("SAMPLE OUTPUT")
print("="*50)

print(

    df[
        [
            "resume_experience",
            "jd_experience",
            "experience_match_score"
        ]
    ].head(10)

)

# ============================================================
# STEP 8 : STATISTICS
# ============================================================

print("\n" + "="*50)
print("EXPERIENCE STATISTICS")
print("="*50)

print(

    df[
        [
            "resume_experience",
            "jd_experience",
            "experience_match_score"
        ]
    ].describe()

)

print(

    "\nMissing Resume Experience:",
    df["resume_experience"]
      .isna()
      .sum()

)

print(

    "Missing JD Experience:",
    df["jd_experience"]
      .isna()
      .sum()

)

# ============================================================
# STEP 9 : CHECK RANDOM ROWS
# ============================================================

print("\n" + "="*50)
print("RANDOM EXPERIENCE CHECK")
print("="*50)

print(

    df[
        [
            "resume_experience",
            "jd_experience",
            "experience_match_score"
        ]
    ].sample(10)

)

# ============================================================
# STEP 10 : SAVE UPDATED DATASET
# ============================================================

df.to_csv(
    "resumeJD_with_experience.csv",
    index=False
)

print("\nDataset Saved Successfully")

print(
    "\nFile Name: resumeJD_with_experience.csv"
)

# ============================================================
# STEP 11 : DOWNLOAD FILE
# ============================================================

from google.colab import files

files.download(
    "resumeJD_with_experience.csv"
)